# Small-scale validation on a real GPU

End-to-end smoke test of the full pipeline on **real DCLM data + a GPU**, before
committing to the expensive 8xA100 run. Downloads a few shards, shrinks the
config to ~100 steps, runs `scripts/train.py`, and checks that:

- train loss drops from ~ln(vocab) and breaks the unigram wall,
- validation loss prints,
- a checkpoint is written and `log/log.txt` is populated.

**Prerequisites:** GPU runtime (Runtime -> Change runtime type -> GPU) and an
`HF_TOKEN` Colab secret (key icon in the left sidebar) with read access to the
private dataset.

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Code

In [3]:
![ -d /content/reproduce-gpt2 ] || git clone -q https://github.com/zzkai098/reproduce-gpt2.git /content/reproduce-gpt2
%cd /content/reproduce-gpt2
!git pull -q
!pip install -q -e . huggingface_hub    # installs the gpt2 package (importable) + base deps

/content/reproduce-gpt2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for reproduce-gpt2 (pyproject.toml) ... done


## 2. Data — download val + a couple train shards from the HF Hub

In [6]:
from huggingface_hub import login, snapshot_download

login()  

snapshot_download(
    repo_id="zzkai098/dclm-gpt2-shards",
    repo_type="dataset",
    local_dir="data/dclm_10B",
    allow_patterns=["dclm_val_000000.npy", "dclm_train_00000[1-2].npy"],  # val + 2 train (~600MB)
)
!ls -lh data/dclm_10B/

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

total 573M
-rw-r--r-- 1 root root 191M Jul 28 02:16 dclm_train_000001.npy
-rw-r--r-- 1 root root 191M Jul 28 02:16 dclm_train_000002.npy
-rw-r--r-- 1 root root 191M Jul 28 02:16 dclm_val_000000.npy


## 3. Shrink the config for a quick validation run

Small batch + 100 steps + torch.compile off. (The real run keeps the full
524288 / B=16,T=1024 / 19073-step config.)

In [7]:
!sed -i 's/^total_batch_size = 524288.*/total_batch_size = 32768/' scripts/train.py
!sed -i 's/^B, T = 16, 1024.*/B, T = 8, 512/' scripts/train.py
!sed -i 's/^model = torch.compile(model)/# model = torch.compile(model)/' scripts/train.py
!sed -i 's/^warmup_steps = 715.*/warmup_steps = 10/' scripts/train.py
!sed -i 's/^max_steps = 19073.*/max_steps = 100/' scripts/train.py
!sed -i 's/^eval_interval = 100/eval_interval = 20/' scripts/train.py
!sed -i 's/^save_every = 5000/save_every = 50/' scripts/train.py
!grep -nE '^total_batch_size|^B, T|torch.compile|^warmup_steps|^max_steps|^eval_interval|^save_every' scripts/train.py

60:save_every = 50
69:total_batch_size = 32768
70:B, T = 8, 512
83:# model = torch.compile(model)
90:warmup_steps = 10
91:max_steps = 100
102:eval_interval = 20


## 4. Train

In [8]:
!python scripts/train.py

total desired batch size: 32768
grad_accum_steps: 8
found 2 shards for split train
found 1 shards for split val
decayed tensors: 50, 124,354,560 params
non-decayed tensors: 98, 121,344 params
using fused AdamW: True
step 0 | val loss 10.9429
step    0 | loss 10.9560 | norm 14.1769 | lr 6.00e-05 | dt 20528ms | tok/s 1596
step    1 | loss 10.0679 | norm 7.0464 | lr 1.20e-04 | dt 10449ms | tok/s 3136
step    2 | loss 9.7906 | norm 2.9866 | lr 1.80e-04 | dt 10597ms | tok/s 3092
step    3 | loss 9.5934 | norm 2.4168 | lr 2.40e-04 | dt 10779ms | tok/s 3040
step    4 | loss 8.6729 | norm 3.9257 | lr 3.00e-04 | dt 10979ms | tok/s 2985
step    5 | loss 10.2718 | norm 24.7804 | lr 3.60e-04 | dt 11198ms | tok/s 2926
step    6 | loss 9.2678 | norm 2.9813 | lr 4.20e-04 | dt 11386ms | tok/s 2878
step    7 | loss 8.9389 | norm 1.8662 | lr 4.80e-04 | dt 11607ms | tok/s 2823
step    8 | loss 8.6542 | norm 1.7963 | lr 5.40e-04 | dt 11878ms | tok/s 2759
step    9 | loss 8.5691 | norm 1.5229 | lr 6.00e-04

## 5. Inspect the outputs

In [ ]:
!echo '=== log/ (checkpoints + log.txt) ===' && ls -lh log/
!echo '=== log.txt (first + last lines) ===' && head -6 log/log.txt && echo '...' && tail -6 log/log.txt